In [0]:
from pyspark.sql.functions import col, when, hour

# Load the processed Silver data
raw_df = spark.table("`us-accidents`.silver.accidents_processed")

# preparing data
df_mllib = raw_df.limit(50000) \
    .select(
        col("Distance_mi").cast("double"),
        col("Temperature_F").cast("double"),
        hour(col("Start_Time")).alias("Hour"),
        when(col("Severity") <= 2, 0).otherwise(1).alias("label")
    ).dropna()

print(f"Data ready for training: {df_mllib.count()} rows processed.")

In [0]:
import mlflow
import os
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Offload caching to a Volume to prevent gRPC 1GB overflow
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/us-accidents/default/temp/sparkml_cache'


assembler = VectorAssembler(inputCols=["Distance_mi", "Temperature_F", "Hour"], outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaled")

dt = DecisionTreeClassifier(featuresCol="scaled", labelCol="label")
pipeline = Pipeline(stages=[assembler, scaler, dt])

# Using CrossValidator for hyperparameter tuning
paramGrid = ParamGridBuilder() \
    .addGrid(dt.maxDepth, [3, 5, 7]) \
    .addGrid(dt.minInstancesPerNode, [1, 5, 10]) \
    .build()

evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

crossValidator = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    seed=42,
    parallelism=1
)

with mlflow.start_run(run_name="Phase3_DecisionTree_MLlib"):
    print("Training Decision Tree with CrossValidator...")
    cv_model = crossValidator.fit(df_mllib)
    
    # get the best model details
    best_dt = cv_model.bestModel.stages[-1]
    print(f"Best maxDepth: {best_dt.getMaxDepth()}")
    print(f"Best minInstancesPerNode: {best_dt.getMinInstancesPerNode()}")
    
    # Log the parameters to MLflow for tracking
    mlflow.log_param("maxDepth", best_dt.getMaxDepth())
    mlflow.log_param("minInstancesPerNode", best_dt.getMinInstancesPerNode())
    mlflow.log_param("model_type", "DecisionTree")
    
    print("\nDecision Tree training complete!")

In [0]:
import pandas as pd
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

best_model = cv_model.bestModel
best_dt = best_model.stages[-1]

predictions = best_model.transform(df_mllib)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

metrics_data = {
    "Metric": ["F1 Score", "Accuracy", "Weighted Precision", "Weighted Recall"],
    "Value": [
        round(evaluator.evaluate(predictions, {evaluator.metricName: "f1"}), 4),
        round(evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"}), 4),
        round(evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"}), 4),
        round(evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"}), 4)
    ]
}

print("="*50)
print("             PHASE 3 MODEL EVALUATION             ")
print("="*50)
print(f"Model Architecture:  Decision Tree Classifier")
print(f"Best Max Depth:      {best_dt.getMaxDepth()}")
print(f"Best Min Instances:  {best_dt.getMinInstancesPerNode()}")
print("-" * 50)

results_df = pd.DataFrame(metrics_data)
print(results_df.to_string(index=False))
print("-" * 50)

phase2_f1 = 0.7690 
improvement = metrics_data["Value"][0] - phase2_f1

print(f"Phase 2 F1 Score:    {phase2_f1}")
print(f"Phase 3 F1 Score:    {metrics_data['Value'][0]}")
print(f"Difference:          {round(improvement, 4)}")
print("="*50)

model_path = "/Volumes/us-accidents/default/temp/models/phase3_decision_tree"
best_model.write().overwrite().save(model_path)
print(f"Model saved to: {model_path}")